# 03. Classical Baselines: Majority Category Baseline

Fundamentals of Natural Language / NLP-I, Universitat Autonoma de Barcelona, academic year 2025-2026.

Team 10: Phoebe Iglesias (1713459), David Redrejo (1790336), and Pau Rossell (1750424). Supervisors: Ernest Valveny and Lei Kang.

In this notebook we start the modeling phase with the simplest possible model: always predict the most frequent ICD-10 category prefix in the training split. This model is not designed to win the Kaggle competition. It is a sanity baseline that tells us how much of the validation accuracy can be explained by class imbalance alone.

## Why This Baseline Is Necessary

From the EDA we already saw that the category distribution is imbalanced. After reading the ICD coding survey, we also understood that imbalance is a central problem in automated ICD coding: frequent clinical categories dominate the data, while rare categories are still medically important.

A majority baseline gives us the first honest threshold. If a later model cannot beat it clearly, the model is probably learning the dataset prior rather than useful linguistic or clinical evidence from the literal. Beating this model is therefore the minimum requirement for any meaningful approach.

## What v00 Does

The script `models/v00_majority_baseline.py` follows the shared model-version contract:

1. Load `data/processed/train_required_clean.csv` and `data/processed/leaderboard_required_clean.csv`.
2. Use the existing annotation contract where `y_category` is derived from the first character of `Code`.
3. Create an 80/20 validation split stratified by `label_id` with seed 42.
4. Find the most frequent `y_category` in the training split.
5. Predict that category for every validation and leaderboard example.
6. Save metrics, validation predictions, a Kaggle submission, and a run summary.

In [ ]:
from pathlib import Path
import json
import pandas as pd

metrics_path = Path('../outputs/metrics/v00_majority_baseline_metrics.json')
submission_path = Path('../submissions/v00_majority_baseline_submission.csv')
val_predictions_path = Path('../outputs/predictions/v00_majority_baseline_val_predictions.csv')

metrics = json.loads(metrics_path.read_text())
submission = pd.read_csv(submission_path)
val_predictions = pd.read_csv(val_predictions_path)

results = pd.DataFrame([
    {
        'model': 'v00_majority_baseline',
        'validation_accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'weighted_f1': metrics['weighted_f1'],
        'predicted_category': val_predictions['y_pred'].mode()[0],
        'submission_rows': len(submission),
        'submission_columns': ', '.join(submission.columns),
    }
])
results

## Results

| model | validation accuracy | macro F1 | weighted F1 | predicted category |
|---|---:|---:|---:|---|
| v00_majority_baseline | 0.1252 | 0.0062 | 0.0279 | Z |

The majority category in the training split was `Z`. The model predicted `Z` for every validation example and every leaderboard example. This gives an accuracy of about 12.5%, but macro F1 is almost zero because all non-`Z` classes receive no positive predictions.

This is exactly why the baseline is useful: it shows that class imbalance alone gives a non-trivial accuracy, while also showing that such a model is clinically and linguistically empty.

In [ ]:
assert submission.columns.tolist() == ['id', 'y_category']
assert len(submission) == 6667
submission.head()

## Interpretation for the Next Step

Our next classical baselines must beat 12.5% validation accuracy and, more importantly, must improve macro F1 by predicting more than one category. TF-IDF character n-grams are the next reasonable step because clinical literals contain compact morphology, abbreviations, punctuation, digits, and partial code-like patterns that may be informative even before using RoBERTa.

For the report, this result will be presented as the lower bound of the modeling section: a model that knows only the class prior and ignores the literal text.